# 04 - Weekly baseline v2 (history and graph features)

Goal: a stronger and fair tabular opponent for the later GNN. Every feature for week *w* uses only data up to *w*: **history features** use strictly earlier weeks (mean over the user's previous 4 active weeks, deviation of this week from it, number of PCs never used before); **graph statistics** use the same week only (how many other users logged on to the user's PCs that week). Predictions are made **at the end of each week**, so all of week *w* is known when week *w* is scored.

Because history needs 4 earlier active weeks, each user's **first 4 active weeks are removed for every model**, including the old five-count features, so all comparisons use identical rows. Feature sets: **W1** = five counts, **W2** = W1 + history, **W3** = W1 + graph, **W4** = W1 + both. Same 25 user-level folds, same fixed hyperparameters as notebook 03, tie-aware metrics. No charts in this notebook, only tables.

In [1]:
%load_ext autoreload
%autoreload 2
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from xgboost import XGBClassifier

from itgnn.data import load_config, prepare
from itgnn.evaluation import make_folds, evaluate_scores
from itgnn.weekly_features import (DEV_COLS, FEATURE_SETS, GRAPH_COLS, HIST_COLS, add_graph_stats,
                                   add_history_features, drop_warmup, signed_log1p)
from itgnn.windows import WEEK_FEATURE_COLS, load_attack_windows, label_weeks, make_weekly_features, week_masks

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
cfg = load_config("configs/config.yaml")
SEED, QS = cfg["seed"], [0.25, 0.5, 1, 2]
pd.options.display.float_format = "{:,.3f}".format
pd.options.display.width = 200
pd.options.display.max_columns = 50
pd.options.display.max_rows = 200

d = prepare(cfg)
users, labels = d["users"], d["labels"]
folds = make_folds(users, labels, cfg["cv"]["n_splits"], cfg["cv"]["n_repeats"], SEED)
windows = load_attack_windows(cfg["raw_dir"])
ins = pd.read_csv(Path(cfg["raw_dir"]) / "answers" / "insiders.csv")
scen = ins[ins["dataset"].astype(str) == "4.2"].set_index("user")["scenario"]
SCEN = sorted(scen.unique())

full = make_weekly_features(d["logon"], d["device"], d["file"], users)
full["label"] = label_weeks(full, windows)
# diagnostic controls are computed on the FULL data (before the warm-up drop), as in notebook 03
origin = full["week_start"].min()
full["cal_week"] = (full["week_start"] - origin).dt.days // 7
full["weeks_since_first"] = (full["week_start"] - full.groupby("user")["week_start"].transform("min")).dt.days // 7
full["weeks_until_last"] = (full.groupby("user")["week_start"].transform("max") - full["week_start"]).dt.days // 7
full = add_graph_stats(add_history_features(full, d["logon"]), d["logon"])
weekly = drop_warmup(full)

## 1. What the warm-up removes

Rows and attack weeks lost, and how many insiders no longer have any attack week left (**eligible** insiders = at least one attack week remains). Insiders lost to the warm-up stay in the folds (their remaining weeks are normal-labelled rows) but cannot be caught, so headline catch counts use the eligible insiders as the denominator.

In [2]:
print(f"rows: {len(full):,} -> {len(weekly):,} (removed {len(full)-len(weekly):,}, {1-len(weekly)/len(full):.1%})")
print(f"attack weeks: {int(full['label'].sum())} -> {int(weekly['label'].sum())} (removed {int(full['label'].sum()-weekly['label'].sum())}); prevalence {full['label'].mean():.4f} -> {weekly['label'].mean():.4f}")
print(f"users with >=1 row: {full['user'].nunique()} -> {weekly['user'].nunique()}")
aw_full = full[full["label"] == 1].groupby("user").size().reindex(list(labels), fill_value=0)
aw_rest = weekly[weekly["label"] == 1].groupby("user").size().reindex(list(labels), fill_value=0)
eligible = set(aw_rest[aw_rest > 0].index)
tab = pd.DataFrame({"insiders": scen.value_counts(), "eligible": scen[list(eligible)].value_counts(),
                    "attack weeks before": aw_full.groupby(scen).sum(), "attack weeks after": aw_rest.groupby(scen).sum()}).fillna(0).astype(int)
tab["insiders lost"] = tab["insiders"] - tab["eligible"]
tab.loc["all"] = tab.sum()
display(tab)
ELIG = {s: sorted(u for u in eligible if scen[u] == s) for s in SCEN}
NORMALS = [u for u in weekly["user"].unique() if u not in labels]
print(f"eligible insiders: {len(eligible)} of {len(labels)} | normal users with rows after the warm-up: {len(NORMALS)} of {len(users)-len(labels)}")
y = weekly["label"]
FEATS = sorted({c for s in FEATURE_SETS.values() for c in s})
assert np.isfinite(weekly[FEATS].to_numpy(float)).all(), "NaN or inf in the feature matrix"
display(weekly[HIST_COLS + GRAPH_COLS].describe().T)
# n_new_pcs is inflated while history is short: a user with few weeks sees more PCs as new
rk = full.groupby("user").cumcount()
bk = pd.cut(rk, [-1, 3, 7, 11, 19, 39, 1000], labels=["0-3 (dropped)", "4-7", "8-11", "12-19", "20-39", "40+"])
display(full.groupby(bk, observed=True)["n_new_pcs"].agg(mean="mean", share_with_new_pc=lambda s: (s > 0).mean()).rename_axis("active-week index"))

rows: 67,238 -> 63,238 (removed 4,000, 5.9%)
attack weeks: 322 -> 322 (removed 0); prevalence 0.0048 -> 0.0051
users with >=1 row: 1000 -> 1000


,insiders,eligible,attack weeks before,attack weeks after,insiders lost
scenario,,,,,
1,30,30,52,52,0
2,30,30,260,260,0
3,10,10,10,10,0
all,70,70,322,322,0


eligible insiders: 70 of 70 | normal users with rows after the warm-up: 930 of 930


,count,mean,std,min,25%,50%,75%,max
hist_mean_n_logon,"63,238.000",7.067,3.614,4.000,5.000,5.000,9.000,30.000
hist_mean_n_after_hours,"63,238.000",0.472,2.085,0.000,0.000,0.000,0.000,17.250
hist_mean_n_pcs,"63,238.000",1.673,2.710,1.000,1.000,1.000,1.250,24.250
hist_mean_n_device,"63,238.000",6.073,18.283,0.000,0.000,0.000,0.000,156.250
hist_mean_n_file,"63,238.000",6.681,21.680,0.000,0.000,0.000,0.000,178.500
dev_n_logon,"63,238.000",-0.080,1.438,-25.000,-0.250,0.000,0.250,12.750
dev_n_after_hours,"63,238.000",-0.005,0.717,-13.250,0.000,0.000,0.000,12.750
dev_n_pcs,"63,238.000",-0.007,0.794,-18.500,0.000,0.000,0.000,12.750
dev_n_device,"63,238.000",-0.053,3.599,-148.000,0.000,0.000,0.000,40.000
dev_n_file,"63,238.000",-0.067,5.965,-145.750,0.000,0.000,0.000,99.750


,mean,share_with_new_pc
active-week index,,
0-3 (dropped),0.917,0.325
4-7,0.591,0.067
8-11,0.544,0.062
12-19,0.478,0.056
20-39,0.355,0.052
40+,0.196,0.045


## 2. Models and cross-validation

Hyperparameters are exactly those of notebook 03 (fixed, not tuned); class weights are recomputed from every training fold. Logistic regression uses `log1p` on non-negative columns and a signed log `sign(x)*log1p(|x|)` on the `dev_*` columns, then standardisation. Every feature matrix passed to a model is checked to be finite. Controls are small random forests on one column, as in 03, on the restricted rows.

In [3]:
def lr(cols):
    dev = [c for c in cols if c in DEV_COLS]
    pos = [c for c in cols if c not in DEV_COLS]
    tr = [("pos", FunctionTransformer(np.log1p), pos)] + ([("dev", FunctionTransformer(signed_log1p), dev)] if dev else [])
    return make_pipeline(ColumnTransformer(tr), StandardScaler(), LogisticRegression(C=1.0, class_weight="balanced", max_iter=2000))
def rf():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=3, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)
def small_rf():
    return RandomForestClassifier(n_estimators=100, min_samples_leaf=20, class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)
def xgb(ytr):
    return XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1, scale_pos_weight=(ytr == 0).sum() / (ytr == 1).sum(),
                         n_jobs=4, random_state=SEED, verbosity=0)
fit_score = lambda m, Xtr, ytr, Xte: m.fit(Xtr, ytr).predict_proba(Xte)[:, 1]

MODELS = {  # name -> (columns, scorer)
    "constant prior": (WEEK_FEATURE_COLS, lambda Xtr, ytr, Xte: np.full(len(Xte), ytr.mean())),
    "n_device > 0": (WEEK_FEATURE_COLS, lambda Xtr, ytr, Xte: (Xte["n_device"] > 0).to_numpy(float)),
}
ALGOS = {"LR": lambda cols, ytr: lr(cols), "RF": lambda cols, ytr: rf(), "XGB": lambda cols, ytr: xgb(ytr)}
for s, cols in FEATURE_SETS.items():
    for a, mk in ALGOS.items():
        MODELS[f"{a} {s}"] = (cols, (lambda mk, cols: lambda Xtr, ytr, Xte: fit_score(mk(cols, ytr), Xtr, ytr, Xte))(mk, cols))
for c, nm in [("cal_week", "control: calendar week"), ("weeks_since_first", "control: weeks since first"),
              ("weeks_until_last", "control: weeks until last (future)")]:
    MODELS[nm] = ([c], lambda Xtr, ytr, Xte: fit_score(small_rf(), Xtr, ytr, Xte))
ORDER = list(MODELS)
metric_rows, oof_parts = [], []
for f in folds:
    tr, te = week_masks(weekly, f)
    for name, (cols, fn) in MODELS.items():
        Xtr, Xte = weekly.loc[tr, cols], weekly.loc[te, cols]
        assert np.isfinite(Xtr.to_numpy(float)).all() and np.isfinite(Xte.to_numpy(float)).all(), name
        s = fn(Xtr, y[tr], Xte)
        assert np.isfinite(s).all(), name
        ev = evaluate_scores(y[te].to_numpy(), s, k=100)
        metric_rows.append({"model": name, "repeat": f["repeat"], "fold": f["fold"], "pr_auc": ev["pr_auc"], "roc_auc": ev["roc_auc"]})
        oof_parts.append(pd.DataFrame({"user": weekly.loc[te, "user"].to_numpy(), "week_start": weekly.loc[te, "week_start"].to_numpy(),
                                       "repeat": f["repeat"], "fold": f["fold"], "model": name, "score": s, "label": y[te].to_numpy()}))
results = pd.DataFrame(metric_rows)
oof = pd.concat(oof_parts, ignore_index=True)
oof.to_parquet("results/weekly_v2_oof_scores.parquet", index=False)
print("saved results/weekly_v2_oof_scores.parquet:", oof.shape, "| prevalence:", round(y.mean(), 4))

saved results/weekly_v2_oof_scores.parquet: (5375230, 7) | prevalence: 0.0051


## 3. Results: week-level PR-AUC and ROC-AUC

Mean ± std over the 25 folds, on the restricted rows. PR-AUC of the constant prior equals the prevalence of the restricted data.

In [4]:
S = results.groupby("model")[["pr_auc", "roc_auc"]].agg(["mean", "std"]).loc[ORDER]
display(pd.DataFrame({m.replace("_", "-").upper(): S[m].apply(lambda r: f"{r['mean']:.3f} ± {r['std']:.3f}", axis=1) for m in ["pr_auc", "roc_auc"]}))
n3 = {"LR": (0.054, 0.024), "RF": (0.283, 0.071), "XGB": (0.343, 0.087)}  # notebook 03 (all rows), from docs/PROJECT_LOG.md
print("W1 on restricted rows vs notebook 03 (all rows) PR-AUC: " + "; ".join(
    f"{a} {S.loc[f'{a} W1', ('pr_auc','mean')]:.3f} ± {S.loc[f'{a} W1', ('pr_auc','std')]:.3f} vs {m:.3f} ± {sd:.3f}" for a, (m, sd) in n3.items()))

,PR-AUC,ROC-AUC
model,,
constant prior,0.005 ± 0.001,0.500 ± 0.000
n_device > 0,0.024 ± 0.004,0.896 ± 0.013
LR W1,0.058 ± 0.025,0.953 ± 0.012
RF W1,0.297 ± 0.075,0.969 ± 0.025
XGB W1,0.358 ± 0.091,0.982 ± 0.008
LR W2,0.088 ± 0.058,0.963 ± 0.014
RF W2,0.503 ± 0.071,0.982 ± 0.013
XGB W2,0.458 ± 0.080,0.986 ± 0.012
LR W3,0.056 ± 0.025,0.951 ± 0.013


W1 on restricted rows vs notebook 03 (all rows) PR-AUC: LR 0.058 ± 0.025 vs 0.054 ± 0.024; RF 0.297 ± 0.075 vs 0.283 ± 0.071; XGB 0.358 ± 0.091 vs 0.343 ± 0.087


## 4. Paired differences (PR-AUC)

Same folds for every model, so we subtract fold by fold. Repeated-CV folds are correlated (shared users and training data), so the std understates the true uncertainty: descriptive only, not a significance test.

In [5]:
pv = results.pivot_table(index=["repeat", "fold"], columns="model", values="pr_auc")
rows = {}
for a in ["RF", "XGB"]:
    for hi, lo in [("W2", "W1"), ("W3", "W1"), ("W4", "W1"), ("W4", "W2")]:
        dl = pv[f"{a} {hi}"] - pv[f"{a} {lo}"]
        rows[f"{a}: {hi} - {lo}"] = {"mean diff": dl.mean(), "std": dl.std(), "folds where first wins": f"{(dl > 0).mean():.0%}"}
display(pd.DataFrame(rows).T)

,mean diff,std,folds where first wins
RF: W2 - W1,0.206,0.064,100%
RF: W3 - W1,-0.011,0.036,48%
RF: W4 - W1,0.196,0.057,100%
RF: W4 - W2,-0.011,0.025,40%
XGB: W2 - W1,0.100,0.074,92%
XGB: W3 - W1,-0.011,0.027,28%
XGB: W4 - W1,0.101,0.061,92%
XGB: W4 - W2,0.001,0.025,60%


## 5. Alert budget

In each test fold flag the top q% of the restricted user-weeks by score (ties broken at random, fixed seed). Per repeat, pool the 5 test folds so every user appears once. **Insiders caught** = eligible insiders with at least one attack week flagged (denominators in the header lines); **normal users flagged** = distinct normal users with any week flagged. Mean ± std over the 5 repeats. Flagging the top q% over the whole period is easier than a real per-week workflow.

In [6]:
user_scen = scen.to_dict()
rows = []
for (name, r), g in oof.groupby(["model", "repeat"]):
    rng = np.random.default_rng(SEED + r)
    parts = {q: [] for q in QS}
    for _, gf in g.groupby("fold"):
        order = np.lexsort((rng.random(len(gf)), -gf["score"].to_numpy()))
        for q in QS:
            parts[q].append(gf.iloc[order[: int(np.ceil(q / 100 * len(gf)))]])
    for q in QS:
        fl = pd.concat(parts[q])
        att = fl[fl["label"] == 1].assign(scenario=lambda x: x["user"].map(user_scen))
        row = {"model": name, "q%": q, "repeat": r, "caught": att["user"].nunique(),
               "normal flagged": fl.loc[~fl["user"].isin(labels), "user"].nunique(), "weeks flagged": len(fl)}
        for s in SCEN:
            row[f"caught s{s}"] = att.loc[att["scenario"] == s, "user"].nunique()
            row[f"week recall s{s}"] = (att["scenario"] == s).sum() / tab.loc[s if s in tab.index else str(s), "attack weeks after"]
        rows.append(row)
bud = pd.DataFrame(rows)
print(f"eligible insiders: {len(eligible)} total; per scenario " + ", ".join(f"s{s}: {len(ELIG[s])}" for s in SCEN) + f" | normal users: {len(NORMALS)}")
agg = bud.groupby(["model", "q%"]).agg(["mean", "std"])
fmt = lambda c: agg[c].apply(lambda r: f"{r['mean']:.1f} ± {r['std']:.1f}", axis=1)
display(pd.DataFrame({f"caught / {len(eligible)}": fmt("caught"), "normal flagged": fmt("normal flagged"), "weeks flagged": fmt("weeks flagged")}).reindex(ORDER, level=0))

eligible insiders: 70 total; per scenario s1: 30, s2: 30, s3: 10 | normal users: 930


caught / 70 normal flagged weeks flagged
model                              q%                                            
constant prior                     0.250   0.6 ± 0.9    142.8 ± 1.3   160.4 ± 0.5
                                   0.500   1.4 ± 1.5    258.8 ± 8.4   318.8 ± 0.4
                                   1.000   3.6 ± 1.9    443.8 ± 8.8   634.6 ± 0.9
                                   2.000   5.4 ± 2.1    672.0 ± 6.4  1267.2 ± 0.8
n_device > 0                       0.250   3.8 ± 1.9     99.2 ± 2.7   160.4 ± 0.5
                                   0.500   7.2 ± 2.3    142.6 ± 4.7   318.8 ± 0.4
                                   1.000  12.2 ± 2.6    177.2 ± 2.2   634.6 ± 0.9
                                   2.000  20.8 ± 2.3    188.4 ± 1.9  1267.2 ± 0.8
LR W1                              0.250   2.6 ± 2.1     14.2 ± 2.2   160.4 ± 0.5
                                   0.500   5.2 ± 3.5     17.6 ± 1.5   318.8 ± 0.4
                                   1.000   9.6 ± 4.4     25.2 ± 1.1   634.6 ± 0.9
                                   2.000  27.2 ± 1.3     45.0 ± 5.4  1267.2 ± 0.8
RF W1                              0.250  35.0 ± 1.2     10.0 ± 1.2   160.4 ± 0.5
                                   0.500  40.0 ± 1.0     17.4 ± 4.4   318.8 ± 0.4
                                   1.000  46.2 ± 1.1     35.4 ± 4.4   634.6 ± 0.9
                                   2.000  56.2 ± 2.3     84.2 ± 3.7  1267.2 ± 0.8
XGB W1                             0.250  36.6 ± 3.0     14.6 ± 2.6   160.4 ± 0.5
                                   0.500  43.0 ± 2.0     20.8 ± 4.0   318.8 ± 0.4
                                   1.000  46.4 ± 1.7     27.8 ± 4.8   634.6 ± 0.9
                                   2.000  54.0 ± 1.2     58.0 ± 1.9  1267.2 ± 0.8
LR W2                              0.250   5.0 ± 2.4     14.0 ± 4.2   160.4 ± 0.5
                                   0.500   9.0 ± 2.3     16.4 ± 3.8   318.8 ± 0.4
                                   1.000  17.0 ± 2.2     24.0 ± 2.0   634.6 ± 0.9
                                   2.000  33.8 ± 3.6     47.8 ± 4.9  1267.2 ± 0.8
RF W2                              0.250  51.4 ± 2.3      3.2 ± 0.4   160.4 ± 0.5
                                   0.500  56.6 ± 1.7      9.2 ± 1.1   318.8 ± 0.4
                                   1.000  61.2 ± 0.8     18.2 ± 3.6   634.6 ± 0.9
                                   2.000  65.4 ± 0.9     35.6 ± 4.6  1267.2 ± 0.8
XGB W2                             0.250  48.0 ± 3.1      2.8 ± 0.8   160.4 ± 0.5
                                   0.500  52.6 ± 0.9      5.4 ± 0.5   318.8 ± 0.4
                                   1.000  56.6 ± 1.3     10.2 ± 1.8   634.6 ± 0.9
                                   2.000  62.6 ± 1.3     30.2 ± 6.8  1267.2 ± 0.8
LR W3                              0.250   2.6 ± 2.1     14.2 ± 2.0   160.4 ± 0.5
                                   0.500   5.4 ± 3.2     17.0 ± 0.7   318.8 ± 0.4
                                   1.000  10.0 ± 3.9     25.4 ± 1.5   634.6 ± 0.9
                                   2.000  27.0 ± 2.4     45.6 ± 4.6  1267.2 ± 0.8
RF W3                              0.250  32.8 ± 2.3     10.2 ± 1.3   160.4 ± 0.5
                                   0.500  36.4 ± 2.5     17.2 ± 3.8   318.8 ± 0.4
                                   1.000  44.4 ± 1.9     35.2 ± 1.9   634.6 ± 0.9
                                   2.000  54.2 ± 2.2     76.8 ± 7.4  1267.2 ± 0.8
XGB W3                             0.250  33.8 ± 2.4     15.8 ± 2.7   160.4 ± 0.5
                                   0.500  40.4 ± 2.1     21.6 ± 3.0   318.8 ± 0.4
                                   1.000  44.8 ± 3.1     29.0 ± 4.1   634.6 ± 0.9
                                   2.000  54.4 ± 2.7     60.0 ± 3.5  1267.2 ± 0.8
LR W4                              0.250   5.4 ± 2.6     14.4 ± 3.8   160.4 ± 0.5
                                   0.500   9.6 ± 2.9     17.4 ± 3.8   318.8 ± 0.4
                                   1.000  17.4 ± 2.3     25.8 ± 3.1   634.6 ± 0.9
                       

## 6. Per scenario

Insiders caught per scenario (mean over 5 repeats; denominators = eligible insiders of that scenario) and **week-level recall** per scenario (share of that scenario's remaining attack weeks that are flagged, mean over repeats), for the three algorithms on each feature set. Scenario 2 holds most attack weeks, so overall PR-AUC mostly reflects scenario 2.

In [7]:
KEY = [f"{a} {s}" for a in ["LR", "RF", "XGB"] for s in FEATURE_SETS] + ["n_device > 0"]
m = bud.groupby(["model", "q%"]).mean(numeric_only=True)
cau = m[[f"caught s{s}" for s in SCEN]].unstack("q%").reindex(KEY)
print("insiders caught per scenario (eligible: " + ", ".join(f"s{s}={len(ELIG[s])}" for s in SCEN) + ")")
display(cau.swaplevel(axis=1).sort_index(axis=1, level=[0], sort_remaining=True).reindex(columns=[(q, f"caught s{s}") for s in SCEN for q in QS]))
rec = m[[f"week recall s{s}" for s in SCEN]].unstack("q%").reindex(KEY)
print("week-level recall per scenario")
display(rec.swaplevel(axis=1).reindex(columns=[(q, f"week recall s{s}") for s in SCEN for q in QS]))

insiders caught per scenario (eligible: s1=30, s2=30, s3=10)


q%,0.250,0.500,1.000,2.000,0.250,0.500,1.000,2.000,0.250,0.500,1.000,2.000
,caught s1,caught s1,caught s1,caught s1,caught s2,caught s2,caught s2,caught s2,caught s3,caught s3,caught s3,caught s3
model,,,,,,,,,,,,
LR W1,1.400,2.600,4.400,8.800,1.000,2.000,4.200,16.600,0.200,0.600,1.000,1.800
LR W2,3.800,6.000,9.800,17.200,1.200,3.000,7.000,15.800,0.000,0.000,0.200,0.800
LR W3,1.200,2.600,4.400,8.400,1.000,2.000,4.600,16.800,0.400,0.800,1.000,1.800
LR W4,4.400,7.000,10.000,19.200,1.000,2.600,7.000,17.200,0.000,0.000,0.400,1.200
RF W1,11.400,13.600,17.000,23.000,22.400,23.600,24.400,26.600,1.200,2.800,4.800,6.600
RF W2,23.000,27.000,28.800,29.600,23.400,24.000,26.400,29.200,5.000,5.600,6.000,6.600
RF W3,10.000,11.600,15.600,22.600,22.600,23.800,24.600,25.800,0.200,1.000,4.200,5.800
RF W4,21.200,25.800,28.000,29.600,23.000,23.200,24.600,28.600,5.000,5.600,5.800,6.400


week-level recall per scenario


q%,0.250,0.500,1.000,2.000,0.250,0.500,1.000,2.000,0.250,0.500,1.000,2.000
,week recall s1,week recall s1,week recall s1,week recall s1,week recall s2,week recall s2,week recall s2,week recall s2,week recall s3,week recall s3,week recall s3,week recall s3
model,,,,,,,,,,,,
LR W1,0.027,0.050,0.088,0.219,0.005,0.013,0.032,0.164,0.020,0.060,0.100,0.180
LR W2,0.073,0.123,0.227,0.435,0.008,0.019,0.069,0.273,0.000,0.000,0.020,0.080
LR W3,0.023,0.050,0.088,0.208,0.005,0.012,0.034,0.165,0.040,0.080,0.100,0.180
LR W4,0.085,0.142,0.235,0.469,0.007,0.022,0.074,0.277,0.000,0.000,0.040,0.120
RF W1,0.269,0.342,0.458,0.654,0.196,0.302,0.471,0.668,0.120,0.280,0.480,0.660
RF W2,0.531,0.715,0.858,0.938,0.305,0.416,0.552,0.711,0.500,0.560,0.600,0.660
RF W3,0.219,0.285,0.427,0.646,0.206,0.332,0.485,0.678,0.020,0.100,0.420,0.580
RF W4,0.504,0.677,0.827,0.923,0.299,0.425,0.535,0.712,0.500,0.560,0.580,0.640


## Key takeaways

- **Warm-up.** Dropping each user's first 4 active weeks removes 4,000 of 67,238 rows (5.9%) but **0 attack weeks and 0 insiders**: all 70 insiders (s1: 30, s2: 30, s3: 10) stay eligible, so every catch count below is out of 70, and 930 normal users remain. Prevalence moves from 0.48% to 0.51%. W1 on the restricted rows is slightly higher than notebook 03 (RF 0.297 vs 0.283, XGB 0.358 vs 0.343, LR 0.058 vs 0.054), within the fold spread.
- **History features help a lot; graph statistics do not.** PR-AUC, W1 -> W2: RF 0.297 -> 0.503 (paired diff +0.206 ± 0.064, W2 better in 25/25 folds), XGBoost 0.358 -> 0.458 (+0.100 ± 0.074, 92% of folds). Logistic regression gains little (0.058 -> 0.088, still weak). W3 - W1 is -0.011 ± 0.036 (RF) and -0.011 ± 0.027 (XGBoost); W4 - W2 is -0.011 (RF) and +0.001 (XGBoost), i.e. no gain from the three same-week graph statistics. RF W2 (0.503) vs XGBoost W2 (0.458): spreads overlap and I did not test it, so I would not rank them. Paired stds understate uncertainty (correlated folds).
- **Alert budget (mean over 5 repeats, out of 70 insiders / 930 normal users).** At 0.25% (~160 weeks): RF W1 catches 35.0 and flags 10.0 normal users; RF W2 catches 51.4 and flags 3.2; XGBoost W2 48.0 and 2.8. At 2% (~1,267 weeks): RF W1 56.2 / 84.2, RF W2 65.4 / 35.6, XGBoost W2 62.6 / 30.2. So history features both catch more insiders and cost fewer false users. `n_device > 0` at 2% catches 20.8 and flags 188.4; the future-looking control catches 54 but flags 922 normal users.
- **Per scenario.** The gain from history is largest in scenario 1: RF catches 11.4 of 30 at 0.25% with W1 and 23.0 with W2 (week recall 0.27 -> 0.53); at 2% 23.0 -> 29.6. Scenario 2 was already well caught by W1 (RF 22.4 of 30 at 0.25%, 23.4 with W2), but week-level recall rises (0.20 -> 0.31 at 0.25%, 0.67 -> 0.71 at 2%). Scenario 3 has only 10 insiders: RF goes 1.2 -> 5.0 at 0.25%, XGBoost only 3.2 -> 3.8, and LR W2 drops to 0, so I would not read anything into scenario 3 beyond "small and noisy".
- **Controls unchanged.** Calendar week 0.009 and weeks-since-first 0.009 PR-AUC (ROC-AUC about 0.70), future-looking control 0.044 (ROC-AUC 0.938), as in notebook 03 (computed on the full data before the drop). The real models are far above the first two. The leaving-related leak cannot be excluded: a change from a user's own recent behaviour (W2) is also what leaving or preparing to leave could look like, and I did not separate this.
- **`n_new_pcs` is inflated just after the warm-up.** Mean new PCs per week is 0.59 in active weeks 4-7 (6.7% of weeks have at least one) against 0.20 from week 40 on (4.5%), and 0.92 inside the dropped first 4 weeks. Shorter history makes more PCs look new, so part of this feature is a "young account" signal. I did not run an ablation without `n_new_pcs` (or only one history feature), so how much of the W2 gain comes from it is unknown.
- **Limitations.**
  - Predictions are made at the **end of each week**: all of week *w* is known when it is scored, so this is not a within-week or real-time detector.
  - Flagging the top q% over the whole period is easier than a real per-week analyst budget.
  - A week is malicious if it overlaps the attack window at all, and windows last up to 58 days, so labels are noisy and cap precision.
  - About 14 insiders per test fold and correlated repeated folds: stds understate uncertainty; the budget stds are over only 5 repeats.
  - Hyperparameters are fixed, not tuned; the history mean over 4 active weeks is a single arbitrary window; first and last calendar weeks are partial.
  - The 3 graph statistics are simple per-week summaries; they added nothing here, which does not say a GNN on the graph will not.